# 🔴 DBSCAN Clustering
**Module 1 — Clustering Algorithms**

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded ✅')

## 2. What is DBSCAN?
> **Density-Based Spatial Clustering of Applications with Noise**
Finds clusters as **dense regions** separated by low-density areas.

**Key Concepts:**
| Term | Meaning |
|---|---|
| `eps` (ε) | Neighborhood radius |
| `min_samples` | Min points to form a dense region |
| **Core Point** | Has ≥ min_samples within ε |
| **Border Point** | Within ε of core, but fewer neighbors |
| **Noise Point** | Not reachable from any core point → label = **-1** |

**Advantages:** No need to specify K, handles arbitrary shapes, detects outliers

## 3. DBSCAN on Non-Convex Shapes

In [ ]:
datasets = [
    (make_moons(n_samples=300, noise=0.07, random_state=42), 'Moons', 0.15, 5),
    (make_circles(n_samples=300, factor=0.5, noise=0.06, random_state=42), 'Circles', 0.12, 5),
    (make_blobs(n_samples=300, centers=3, cluster_std=0.6, random_state=42), 'Blobs', 0.8, 5),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, ((X, y), name, eps, ms) in enumerate(datasets):
    X_sc = StandardScaler().fit_transform(X)
    # Raw
    axes[0, i].scatter(X_sc[:, 0], X_sc[:, 1], c=y, cmap='tab10', s=20, alpha=0.7)
    axes[0, i].set_title(f'{name} — Raw', fontsize=11, fontweight='bold')
    # DBSCAN
    db = DBSCAN(eps=eps, min_samples=ms)
    labels = db.fit_predict(X_sc)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    colors = ['red' if l == -1 else plt.cm.tab10(l / max(labels.max(), 1)) for l in labels]
    axes[1, i].scatter(X_sc[:, 0], X_sc[:, 1], c=labels, cmap='tab10', s=20, alpha=0.7)
    axes[1, i].set_title(f'{name} — DBSCAN\nClusters={n_clusters} | Noise={n_noise}', fontsize=11, fontweight='bold')

plt.suptitle('DBSCAN — Non-Convex Shape Handling', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. K-Distance Graph — Finding Optimal eps

In [ ]:
X_moons, _ = make_moons(n_samples=300, noise=0.07, random_state=42)
X_sc = StandardScaler().fit_transform(X_moons)
min_samples = 5

neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors.fit(X_sc)
distances, _ = neighbors.kneighbors(X_sc)
distances = np.sort(distances[:, min_samples - 1], axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(distances, lw=2, color='steelblue')
ax.axhline(y=0.15, color='red', linestyle='--', lw=1.5, label='eps = 0.15 (elbow)')
ax.set_xlabel('Points sorted by distance', fontsize=12)
ax.set_ylabel(f'{min_samples}-NN Distance', fontsize=12)
ax.set_title('K-Distance Graph — Optimal eps Selection', fontsize=14, fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()
print('Tip: eps = value at the "elbow" of this curve')

## 5. Effect of eps and min_samples

In [ ]:
X, _ = make_moons(n_samples=300, noise=0.07, random_state=42)
X_sc = StandardScaler().fit_transform(X)

eps_vals = [0.05, 0.15, 0.3, 0.6]
min_samples_vals = [3, 5, 10, 15]

fig, axes = plt.subplots(2, 4, figsize=(22, 9))

for i, eps in enumerate(eps_vals):
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(X_sc)
    n_c = len(set(labels)) - (1 if -1 in labels else 0)
    axes[0, i].scatter(X_sc[:, 0], X_sc[:, 1], c=labels, cmap='tab10', s=15, alpha=0.7)
    axes[0, i].set_title(f'eps={eps}, ms=5\nClusters={n_c}, Noise={(labels==-1).sum()}', fontsize=10)

for i, ms in enumerate(min_samples_vals):
    db = DBSCAN(eps=0.15, min_samples=ms)
    labels = db.fit_predict(X_sc)
    n_c = len(set(labels)) - (1 if -1 in labels else 0)
    axes[1, i].scatter(X_sc[:, 0], X_sc[:, 1], c=labels, cmap='tab10', s=15, alpha=0.7)
    axes[1, i].set_title(f'eps=0.15, ms={ms}\nClusters={n_c}, Noise={(labels==-1).sum()}', fontsize=10)

axes[0, 0].set_ylabel('Varying eps', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Varying min_samples', fontsize=11, fontweight='bold')
plt.suptitle('DBSCAN Parameter Sensitivity', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Core Points, Border Points, Noise Visualization

In [ ]:
X, _ = make_moons(n_samples=300, noise=0.07, random_state=42)
X_sc = StandardScaler().fit_transform(X)

db = DBSCAN(eps=0.15, min_samples=5)
db.fit(X_sc)

core_mask   = np.zeros(len(X_sc), dtype=bool)
core_mask[db.core_sample_indices_] = True
noise_mask  = db.labels_ == -1
border_mask = ~core_mask & ~noise_mask

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(X_sc[core_mask,   0], X_sc[core_mask,   1], c='steelblue', s=40,  label=f'Core Points ({core_mask.sum()})',   alpha=0.8)
ax.scatter(X_sc[border_mask, 0], X_sc[border_mask, 1], c='orange',    s=40,  label=f'Border Points ({border_mask.sum()})', alpha=0.8)
ax.scatter(X_sc[noise_mask,  0], X_sc[noise_mask,  1], c='red',       s=60,  label=f'Noise Points ({noise_mask.sum()})',  marker='x', lw=2)
ax.set_title('DBSCAN — Core / Border / Noise Points', fontsize=14, fontweight='bold')
ax.legend(fontsize=11); plt.tight_layout(); plt.show()

## 7. DBSCAN vs K-Means on Non-Convex Data

In [ ]:
from sklearn.cluster import KMeans
X, _ = make_moons(n_samples=300, noise=0.07, random_state=42)
X_sc = StandardScaler().fit_transform(X)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
datasets_labels = [
    (None, _, 'Ground Truth'),
    (KMeans(n_clusters=2, random_state=42).fit_predict(X_sc), None, 'K-Means (K=2)'),
    (DBSCAN(eps=0.15, min_samples=5).fit_predict(X_sc), None, 'DBSCAN'),
]
for ax, (lbl, true_lbl, title) in zip(axes, datasets_labels):
    c = true_lbl if lbl is None else lbl
    ax.scatter(X_sc[:, 0], X_sc[:, 1], c=c, cmap='tab10', s=25, alpha=0.8)
    ax.set_title(title, fontsize=13, fontweight='bold')
plt.suptitle('DBSCAN vs K-Means on Non-Convex Shapes', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

## 8. Evaluation

In [ ]:
X, _ = make_moons(n_samples=300, noise=0.07, random_state=42)
X_sc = StandardScaler().fit_transform(X)
db   = DBSCAN(eps=0.15, min_samples=5)
labels = db.fit_predict(X_sc)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = (labels == -1).sum()

print('='*45)
print('       DBSCAN Evaluation')
print('='*45)
print(f'  Clusters found : {n_clusters}')
print(f'  Noise points   : {n_noise} ({n_noise/len(labels)*100:.1f}%)')
if n_clusters > 1:
    # Silhouette only on non-noise points
    mask = labels != -1
    sil = silhouette_score(X_sc[mask], labels[mask])
    print(f'  Silhouette     : {sil:.4f} (non-noise only)')
print('='*45)

## 9. Key Takeaways
> - DBSCAN is **shape-agnostic** — works on moons, rings, arbitrary shapes
> - Automatically finds K and **labels outliers** as noise (-1)
> - Two params: **eps** (from K-distance graph) and **min_samples** (usually 2×dims)
> - Struggles with **varying density** clusters (try HDBSCAN for that)